# Cookie Cats — Phase 1 EDA

This notebook fetches/loads the Cookie Cats A/B dataset, runs schema/quality checks, and produces D1/D7 retention visualizations with bootstrap confidence intervals. Artifacts are saved to `data/cookie_cats/` and `reports/figures/` with a concise markdown report in `reports/cookie_cats_eda.md`.


In [11]:
# --- setup
import os
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


for p in [DATA_DIR, REPORTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA_DIR / 'raw.csv'
CLEAN_PARQUET = DATA_DIR / 'clean.parquet'
REPORT_MD = REPORTS_DIR / 'cookie_cats_eda.md'



In [12]:
# --- fetch or load data
from datetime import datetime
import subprocess

DATASET_SLUG = "yufengsui/a-b-test-cookie-cats"
EXPECTED_FILENAME = "cookie_cats.csv"  # common name on Kaggle

if not RAW_CSV.exists():
    # Try Kaggle CLI if credentials exist
    kaggle_user = os.environ.get("KAGGLE_USERNAME")
    kaggle_key = os.environ.get("KAGGLE_KEY")
    if kaggle_user and kaggle_key:
        try:
            print("Attempting Kaggle download…")
            subprocess.run([
                "kaggle", "datasets", "download", DATASET_SLUG, "-p", str(DATA_DIR), "--force"
            ], check=True)
            # Unzip any zip in DATA_DIR
            for f in DATA_DIR.glob("*.zip"):
                subprocess.run(["unzip", "-o", str(f), "-d", str(DATA_DIR)], check=True)
            # If expected filename found, copy to RAW_CSV
            candidates = list(DATA_DIR.glob("*.csv"))
            if candidates:
                # Prefer expected file name if present
                prefer = [c for c in candidates if c.name == EXPECTED_FILENAME]
                src = prefer[0] if prefer else candidates[0]
                src.rename(RAW_CSV)
            else:
                print("No CSV found after download; please place raw CSV at:", RAW_CSV)
        except Exception as e:
            print("Kaggle download failed:", e)
            print("Fallback: manually place the dataset CSV at:", RAW_CSV)
    else:
        print("Kaggle credentials not found. Place the dataset CSV at:", RAW_CSV)
else:
    print("Found existing raw CSV:", RAW_CSV)



Found existing raw CSV: /Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/data/cookie_cats/raw.csv


In [13]:
# --- load + basic schema/quality checks
assert RAW_CSV.exists(), f"Raw CSV not found at {RAW_CSV}."

df = pd.read_csv(RAW_CSV)
print(df.head())
print(df.dtypes)

# standardize column names if needed
cols = {c.lower(): c for c in df.columns}
# expected lowercase names
expected = {
    'userid': 'userid', 'version': 'version', 'sum_gamerounds': 'sum_gamerounds',
    'retention_1': 'retention_1', 'retention_7': 'retention_7'
}
# If mixed case present, rename to lower
rename_map = {}
for want in expected.keys():
    if want not in df.columns.str.lower().tolist():
        # find a matching column by case-insensitive compare
        match = [c for c in df.columns if c.lower() == want]
        if match:
            rename_map[match[0]] = want

if rename_map:
    df = df.rename(columns=rename_map)

# basic checks
summary = {
    'rows': len(df),
    'cols': list(df.columns),
    'duplicates_userid': int(df['userid'].duplicated().sum()) if 'userid' in df else None,
    'missing_counts': df.isna().sum().to_dict(),
}
print(summary)

# enforce types
if 'version' in df:
    df['version'] = df['version'].astype('category')
for c in ['retention_1', 'retention_7']:
    if c in df:
        # accepts 0/1 or True/False
        df[c] = df[c].astype(int)

# class balance
for c in ['retention_1', 'retention_7']:
    if c in df:
        rate = df[c].mean()
        print(f"{c} rate: {rate:.3f}")

# save clean parquet early
df.to_parquet(CLEAN_PARQUET, index=False)
print("Saved:", CLEAN_PARQUET)



   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True
userid             int64
version           object
sum_gamerounds     int64
retention_1         bool
retention_7         bool
dtype: object
{'rows': 90189, 'cols': ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7'], 'duplicates_userid': 0, 'missing_counts': {'userid': 0, 'version': 0, 'sum_gamerounds': 0, 'retention_1': 0, 'retention_7': 0}}
retention_1 rate: 0.445
retention_7 rate: 0.186
Saved: /Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/data/cookie_cats/clean.parquet


In [14]:
# --- helpers: bootstrap CIs and plotting
from typing import Tuple

def bootstrap_proportion_ci(y: pd.Series, n_boot: int = 2000, alpha: float = 0.05, rng_seed: int = 42) -> Tuple[float, float, float]:
    y = y.dropna().astype(int)
    p_hat = float(y.mean())
    if len(y) == 0:
        return float('nan'), float('nan'), float('nan')
    rng = np.random.default_rng(rng_seed)
    boot = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot.append(y.iloc[idx].mean())
    lo = np.quantile(boot, alpha/2)
    hi = np.quantile(boot, 1 - alpha/2)
    return p_hat, float(lo), float(hi)

def bar_with_ci(ax, labels, means, los, his, title, ylabel="Retention rate"):
    x = np.arange(len(labels))
    ax.bar(x, means, color=["#4C78A8" if i==0 else "#72B7B2" for i in range(len(labels))])
    ax.errorbar(x, means, yerr=[np.array(means)-np.array(los), np.array(his)-np.array(means)], fmt='none', ecolor='black', capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel(ylabel)
    ax.set_title(title)




In [15]:
# --- D1/D7 overall + uplift by version
fig_paths = {}

# Overall bars
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, target in enumerate(["retention_1", "retention_7"]):
    p, lo, hi = bootstrap_proportion_ci(df[target])
    bar_with_ci(axes[i], [target.upper()], [p], [lo], [hi], f"Overall {target.upper()} retention")
plt.tight_layout()
path = FIG_DIR / 'cookiecats_overall_d1_d7.png'
fig.savefig(path, dpi=150)
fig_paths['overall'] = path
plt.close(fig)

# Uplift bars by version
if 'version' in df.columns:
    grouped = df.groupby('version')
    labels = []
    means_d1, lo_d1, hi_d1 = [], [], []
    means_d7, lo_d7, hi_d7 = [], [], []
    for v, g in grouped:
        labels.append(str(v))
        for target, means, los, his in [
            ("retention_1", means_d1, lo_d1, hi_d1),
            ("retention_7", means_d7, lo_d7, hi_d7),
        ]:
            p, lo, hi = bootstrap_proportion_ci(g[target])
            means.append(p); los.append(lo); his.append(hi)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bar_with_ci(axes[0], labels, means_d1, lo_d1, hi_d1, "D1 retention by version")
    bar_with_ci(axes[1], labels, means_d7, lo_d7, hi_d7, "D7 retention by version")
    plt.tight_layout()
    path = FIG_DIR / 'cookiecats_uplift_by_version.png'
    fig.savefig(path, dpi=150)
    fig_paths['uplift_by_version'] = path
    plt.close(fig)

# Retention vs sum_gamerounds
if 'sum_gamerounds' in df.columns:
    bins = pd.qcut(df['sum_gamerounds'], q=10, duplicates='drop')
    agg = df.assign(bin=bins).groupby('bin').agg(
        d1_rate=("retention_1", "mean"),
        d7_rate=("retention_7", "mean"),
        n=("userid", "count") if 'userid' in df else ("sum_gamerounds", "count")
    ).reset_index()

    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(range(len(agg)), agg['d1_rate'], marker='o', label='D1')
    ax.plot(range(len(agg)), agg['d7_rate'], marker='o', label='D7')
    ax.set_title('Retention vs sum_gamerounds (deciles)')
    ax.set_xlabel('sum_gamerounds decile (low→high)')
    ax.set_ylabel('Retention rate')
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    path = FIG_DIR / 'cookiecats_retention_vs_gamerounds.png'
    fig.savefig(path, dpi=150)
    fig_paths['retention_vs_gamerounds'] = path
    plt.close(fig)

fig_paths


/var/folders/3x/chcvjhrx0_lgbjl18psgxh440000gn/T/ipykernel_31488/676345382.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('version')
/var/folders/3x/chcvjhrx0_lgbjl18psgxh440000gn/T/ipykernel_31488/676345382.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg = df.assign(bin=bins).groupby('bin').agg(


{'overall': PosixPath('/Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/reports/figures/cookiecats_overall_d1_d7.png'),
 'uplift_by_version': PosixPath('/Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/reports/figures/cookiecats_uplift_by_version.png'),
 'retention_vs_gamerounds': PosixPath('/Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/reports/figures/cookiecats_retention_vs_gamerounds.png')}

In [18]:
# --- write concise EDA report
if REPORT_MD.exists():
    pass

lines = []
lines.append('# Cookie Cats — EDA summary')
lines.append('')
lines.append(f'- Rows: {len(df):,}')
lines.append(f'- Columns: {", ".join(df.columns)}')
if 'retention_1' in df:
    lines.append(f"- D1 retention: {df['retention_1'].mean():.3f}")
if 'retention_7' in df:
    lines.append(f"- D7 retention: {df['retention_7'].mean():.3f}")
if 'version' in df:
    g = df.groupby('version')[['retention_1','retention_7']].mean().reset_index()
    lines.append('- Retention by version:')
    for _, r in g.iterrows():
        lines.append(f"  - {r['version']}: D1={r['retention_1']:.3f}, D7={r['retention_7']:.3f}")

lines.append('')
lines.append('## Figures')
for k, p in fig_paths.items():
    rel = os.path.relpath(p, REPORTS_DIR)
    fname = os.path.splitext(os.path.basename(rel))[0]
    lines.append(f'- {k}: ![{fname}]({rel})')

REPORT_MD.write_text("\n".join(lines))
print('Wrote report:', REPORT_MD)



Wrote report: /Users/ian/WINTERMUTE/6_academia/60_courses/61.14_deeplearning/neural-nets_project/reports/cookie_cats_eda.md


/var/folders/3x/chcvjhrx0_lgbjl18psgxh440000gn/T/ipykernel_31488/3106622295.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby('version')[['retention_1','retention_7']].mean().reset_index()
